In [2]:
from pathlib import Path
import zipfile
import pandas as pd

# ---- CONFIG ----
folder = Path(r"D:\ICST2026_Extension\Feb_16_data\Single_check")
zip_stem = "run_steps_v16_stage3_breakdown"   # file name without extension
target_value = "behnamparsa/toDoList"
column_name = "full_name"
out_path = folder / "behnamparsa_toDoList.csv"
# ---------------

zip_path = folder / f"{zip_stem}.zip"
if not zip_path.exists():
    # fallback if the zip is actually without .zip in the name you gave
    alt = folder / zip_stem
    if alt.exists():
        zip_path = alt
    else:
        raise FileNotFoundError(f"Could not find: {folder / (zip_stem + '.zip')} (or {folder / zip_stem})")

matched_parts = []

with zipfile.ZipFile(zip_path, "r") as z:
    csv_names = [n for n in z.namelist() if n.lower().endswith(".csv") and not n.endswith("/")]
    if not csv_names:
        raise RuntimeError(f"No CSV files found inside {zip_path.name}")

    for csv_name in csv_names:
        with z.open(csv_name) as f:
            df = pd.read_csv(f, low_memory=False)

        if column_name not in df.columns:
            print(f"Skipping (no '{column_name}' column): {csv_name}")
            continue

        part = df[df[column_name].astype(str) == target_value].copy()
        if not part.empty:
            part["_source_csv_in_zip"] = csv_name
            matched_parts.append(part)

# Save output
if matched_parts:
    result = pd.concat(matched_parts, ignore_index=True)
    result.to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"Saved {len(result)} rows to: {out_path}")
else:
    pd.DataFrame().to_csv(out_path, index=False, encoding="utf-8-sig")
    print(f"No matching rows found. Wrote empty file to: {out_path}")


Saved 986 rows to: D:\ICST2026_Extension\Feb_16_data\Single_check\behnamparsa_toDoList.csv
